# EECS E6893 HW2 · Section 1 Streaming Analytics

In [8]:
import time
from datetime import datetime, timedelta, timezone

import requests
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

spark = SparkSession.builder.appName("hw2-streaming").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [9]:
API_KEY = "YOUR_API_KEY"
BASE_URL = "https://api.massive.com"
TICKER = "AAPL"

SIM_START = datetime(2026, 10, 2, 14, 0, tzinfo=timezone.utc)  # past trading day (free tier blocks today)
NUM_PULLS = 7
INTERVAL_MIN = 5
LOOKBACK_MIN = 60
MA_MIN = 30
SLEEP_SECONDS = 300  # wait 5 minutes between pulls

KEY_COLS = ["Stock Name", "UTC Timestamp"]


In [10]:
RAW_SCHEMA = StructType([
    StructField("Stock Name", StringType()),
    StructField("t", LongType()),
    StructField("c", DoubleType()),
    StructField("l", DoubleType()),
    StructField("h", DoubleType()),
    StructField("o", DoubleType()),
    StructField("v", DoubleType()),
])


def get_histdata_massive(ticker, start_dt, end_dt):
    start_ms = int(start_dt.timestamp() * 1000)
    end_ms = int(end_dt.timestamp() * 1000)
    url = f"{BASE_URL}/v2/aggs/ticker/{ticker}/range/1/minute/{start_ms}/{end_ms}"
    params = {"adjusted": "true", "sort": "asc", "limit": 50000, "apiKey": API_KEY}
    response = requests.get(url, params=params, timeout=30)

    if response.status_code == 200:
        results = response.json().get("results", [])
    else:
        print(f"Error {response.status_code}: {response.text}")
        results = []

    rows = [(ticker, int(r["t"]), float(r["c"]), float(r["l"]),
             float(r["h"]), float(r["o"]), float(r["v"])) for r in results]
    df = spark.createDataFrame(rows, RAW_SCHEMA)
    df = df.withColumn("UTC Timestamp", F.expr("timestamp_millis(t)"))
    return df.select("Stock Name", "UTC Timestamp", "c", "l", "h", "o", "v")


def upsert(current_df, new_df, key_cols):
    # left anti join: drop old rows that the new pull replaces, then append the new pull
    if current_df is None:
        return new_df
    kept_old = current_df.join(new_df.select(key_cols), on=key_cols, how="left_anti")
    return kept_old.unionByName(new_df)


def moving_averages(stream_df):
    # 30-minute window ending at each timestamp
    w = (Window.partitionBy("Stock Name")
         .orderBy(F.col("UTC Timestamp").cast("long"))
         .rangeBetween(-(MA_MIN * 60 - 1), 0))
    return stream_df.select(
        F.col("UTC Timestamp").alias("Datetime"),
        F.avg("c").over(w).alias("c_MA"),
        F.avg("l").over(w).alias("l_MA"),
        F.avg("h").over(w).alias("h_MA"),
        F.avg("o").over(w).alias("o_MA"),
        F.avg("v").over(w).alias("v_MA"),
    )


In [ ]:
stream_df = None
ma_df = None

for i in range(NUM_PULLS):
    sim_now = SIM_START + timedelta(minutes=LOOKBACK_MIN + i * INTERVAL_MIN)
    window_start = sim_now - timedelta(minutes=LOOKBACK_MIN)
    wall = datetime.now(timezone.utc).strftime("%H:%M:%S")
    print(f"\n=== Pull {i + 1}/{NUM_PULLS} | real time {wall} UTC | "
          f"data {window_start:%Y-%m-%d %H:%M} -> {sim_now:%H:%M} UTC ===")

    new_df = get_histdata_massive(TICKER, window_start, sim_now)
    print("rows returned by API:", new_df.count())

    stream_df = upsert(stream_df, new_df, KEY_COLS).orderBy("UTC Timestamp").localCheckpoint()

    ma_now = moving_averages(stream_df)
    ma_df = upsert(ma_df, ma_now, ["Datetime"]).orderBy("Datetime").localCheckpoint()

    rng = stream_df.agg(F.min("UTC Timestamp").alias("first"),
                        F.max("UTC Timestamp").alias("last")).collect()[0]
    print(f"stream_df: {stream_df.count()} rows, {rng['first']} -> {rng['last']}")
    ma_df.orderBy(F.desc("Datetime")).show(5, truncate=False)

    if i < NUM_PULLS - 1:
        time.sleep(SLEEP_SECONDS)



=== Pull 1/7 | real time 00:44:59 UTC | data 2026-10-02 14:00 -> 15:00 UTC ===


rows returned by API: 61


stream_df: 61 rows, 2026-10-02 14:00:00 -> 2026-10-02 15:00:00


+-------------------+------------------+------------------+------------------+------------------+------------------+
|Datetime           |c_MA              |l_MA              |h_MA              |o_MA              |v_MA              |
+-------------------+------------------+------------------+------------------+------------------+------------------+
|2026-10-02 15:00:00|333.71867         |333.5910566666667 |333.87066999999996|333.76322         |71983.25341076667 |
|2026-10-02 14:59:00|333.76283666666666|333.63572333333326|333.91446333333334|333.80705333333333|71889.08419333333 |
|2026-10-02 14:58:00|333.8078333333333 |333.68138999999996|333.95313         |333.8448866666667 |63133.21821946667 |
|2026-10-02 14:57:00|333.8456666666667 |333.7130566666666 |333.97513333333336|333.8673866666667 |63364.1735133     |
|2026-10-02 14:56:00|333.8695          |333.73675         |334.00028999999995|333.89812         |63994.523447566666|
+-------------------+------------------+------------------+-----


=== Pull 2/7 | real time 00:50:01 UTC | data 2026-10-02 14:05 -> 15:05 UTC ===


...

rows returned by API: 61


stream_df: 86 rows, 2026-10-02 14:00:00 -> 2026-10-02 15:25:00


+-------------------+------------------+------------------+------------------+------------------+------------------+
|Datetime           |c_MA              |l_MA              |h_MA              |o_MA              |v_MA              |
+-------------------+------------------+------------------+------------------+------------------+------------------+
|2026-10-02 15:25:00|332.02070333333336|331.89786999999995|332.2285633333333 |332.0960566666665 |104957.9932899    |
|2026-10-02 15:24:00|332.0900366666667 |331.96753666666666|332.29823000000005|332.1647233333333 |104551.04464766665|
|2026-10-02 15:23:00|332.15837000000005|332.0332033333333 |332.35423000000003|332.21838999999994|103361.03782686665|
|2026-10-02 15:22:00|332.21037         |332.0895366666666 |332.4115633333334 |332.27738999999997|103083.90843993332|
|2026-10-02 15:21:00|332.2695366666667 |332.1502033333334 |332.47468000000003|332.3462233333333 |104411.61247423332|
+-------------------+------------------+------------------+-----


=== Pull 7/7 | real time 01:15:16 UTC | data 2026-10-02 14:30 -> 15:30 UTC ===


rows returned by API: 61


In [14]:
print("Streamed data:")
stream_df.orderBy("UTC Timestamp").show(200, truncate=False)

print("Moving averages:")
ma_df.orderBy("Datetime").show(200, truncate=False)

print("stream_df rows:", stream_df.count(), "| ma_df rows:", ma_df.count())


Streamed data:


+----------+-------------------+--------+--------+--------+--------+--------------+
|Stock Name|UTC Timestamp      |c       |l       |h       |o       |v             |
+----------+-------------------+--------+--------+--------+--------+--------------+
|AAPL      |2026-10-02 14:00:00|332.08  |332.03  |332.295 |332.23  |63224.170005  |
|AAPL      |2026-10-02 14:01:00|332.59  |332.01  |332.7   |332.09  |82961.02796   |
|AAPL      |2026-10-02 14:02:00|332.725 |332.51  |332.93  |332.61  |113312.099517 |
|AAPL      |2026-10-02 14:03:00|332.71  |332.4801|332.75  |332.72  |56030.927404  |
|AAPL      |2026-10-02 14:04:00|332.7   |332.6   |332.83  |332.676 |56367.402983  |
|AAPL      |2026-10-02 14:05:00|332.895 |332.5863|332.9   |332.72  |88888.204213  |
|AAPL      |2026-10-02 14:06:00|332.935 |332.84  |333.18  |332.895 |116290.141011 |
|AAPL      |2026-10-02 14:07:00|333.37  |332.87  |333.42  |332.935 |100207.335751 |
|AAPL      |2026-10-02 14:08:00|333.6702|333.24  |333.8   |333.35  |182296.6

+-------------------+------------------+------------------+------------------+------------------+------------------+
|Datetime           |c_MA              |l_MA              |h_MA              |o_MA              |v_MA              |
+-------------------+------------------+------------------+------------------+------------------+------------------+
|2026-10-02 14:00:00|332.08            |332.03            |332.295           |332.23            |63224.170005      |
|2026-10-02 14:01:00|332.335           |332.02            |332.4975          |332.15999999999997|73092.5989825     |
|2026-10-02 14:02:00|332.465           |332.18333333333334|332.64166666666665|332.31            |86499.09916066666 |
|2026-10-02 14:03:00|332.52625         |332.257525        |332.66875         |332.4125          |78882.0562215     |
|2026-10-02 14:04:00|332.56100000000004|332.32601999999997|332.70099999999996|332.4652          |74379.12557379999 |
|2026-10-02 14:05:00|332.6166666666667 |332.3694          |332.7

stream_df rows: 91 | ma_df rows: 91
